In [1]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel


In [2]:
from transformers import ClapProcessor, ClapModel


In [3]:
import librosa


In [5]:
import torch

In [6]:
import torch.nn as nn

In [7]:
from torch.utils.data import DataLoader

In [8]:
import numpy as np

In [ ]:
from datasets import load_from_disk

ds = load_from_disk("../01_load_dataset/musiccaps/dataset_audio")

In [9]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

# CLAP (frozen)
clap_processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
clap = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device)
clap.eval()
for p in clap.parameters():
    p.requires_grad = False

# GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
gpt2.train()


/Users/aliozkaya/miniconda3/envs/audio-caption/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [10]:
D_AUDIO = clap.config.projection_dim      # e.g. 512
D_LM = gpt2.config.n_embd                 # 768
PREFIX_LEN = 10

projection = nn.Linear(D_AUDIO, PREFIX_LEN * D_LM).to(device)

In [11]:
optimizer = torch.optim.AdamW(
    list(projection.parameters()) + list(gpt2.parameters()),
    lr=2e-5
)


In [12]:
def get_audio_embedding(sample):
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    if sr != 48000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=48000)

    inputs = clap_processor(
        audios=audio,
        sampling_rate=48000,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        emb = clap.get_audio_features(**inputs)

    return emb  # (1, D_AUDIO)


In [13]:
def train_step(sample):
    audio_emb = get_audio_embedding(sample)     # (1, D_AUDIO)

    # Projection → prefix tokens
    prefix = projection(audio_emb)
    prefix = prefix.view(1, PREFIX_LEN, D_LM)

    # Tokenize caption
    tokens = tokenizer(
        sample["caption"],
        return_tensors="pt"
    ).to(device)

    input_ids = tokens.input_ids                 # (1, T)

    # Text embeddings
    text_embeds = gpt2.transformer.wte(input_ids)

    # Concatenate prefix + text
    inputs_embeds = torch.cat([prefix, text_embeds], dim=1)

    # Labels (ignore prefix)
    labels = input_ids.clone()
    ignore = torch.full((1, PREFIX_LEN), -100, device=device)
    labels = torch.cat([ignore, labels], dim=1)

    outputs = gpt2(
        inputs_embeds=inputs_embeds,
        labels=labels
    )

    return outputs.loss


In [14]:
EPOCHS = 1
N_SAMPLES = 1000   # start small

for epoch in range(EPOCHS):
    total_loss = 0.0

    for i in range(N_SAMPLES):
        optimizer.zero_grad()

        loss = train_step(ds[i])
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (i + 1) % 50 == 0:
            print(f"Step {i+1}, loss = {loss.item():.4f}")

    print(f"Epoch {epoch+1}, avg loss = {total_loss / N_SAMPLES:.4f}")


NameError: name 'ds' is not defined